# `push_to_hub(append=True)`: Three Implementation Approaches

This notebook prototypes the three append strategies under discussion in [huggingface/datasets #6290](https://github.com/huggingface/datasets/issues/6290) and measures their Xet chunk reuse.

**The problem**: a user has a dataset on the Hub and wants to push new rows without re-uploading the unchanged data. The current `push_to_hub` call has no append mode — it always replaces.

**The three approaches**:

| # | Approach | Author | Xet reuse |
|---|----------|--------|----------|
| 1 | **Reshard** — concat + rewrite all shards | current behavior | ~0% |
| 2 | **Add-shard** — keep existing shards, append new shard(s) | Wauplin | ~99% |
| 3 | **CDC-rewrite** — concat + rewrite as single file with CDC row groups | lhoestq | ~97–99% |

We simulate Xet locally using fixed 64KB chunks and Blake2b hashing — the same conservative proxy used in the companion [`parquet-xet-write-strategy`](https://github.com/lex00/parquet-xet) notebook. Real Xet CDC is more resilient to insertions, so actual reuse will be at least this good.

In [1]:
# Colab: uncomment to install
# !pip install pyarrow numpy matplotlib -q

import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import hashlib
import inspect
import os
import shutil
import tempfile

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

# Check whether pyarrow 21's content-defined chunking is available
try:
    sig = inspect.signature(pq.write_table)
    HAS_CDC = 'use_content_defined_chunking' in sig.parameters
except Exception:
    HAS_CDC = False

print(f"pyarrow {pa.__version__}")
print(f"content_defined_chunking available: {HAS_CDC} {'(pyarrow >= 21 required)' if not HAS_CDC else ''}")

pyarrow 23.0.1
content_defined_chunking available: False (pyarrow >= 21 required)


## Xet chunk simulation

The same simulation used in the companion notebook. `compute_chunks` hashes every 64KB block of a file; `dir_delta` counts how many blocks in v2 are absent from v1's hash set — those are the bytes that have to upload.

In [2]:
CHUNK_SIZE = 65_536  # 64 KiB — approximate Xet average chunk size

def compute_chunks(path: str) -> list:
    """Fixed-size chunking with Blake2b — conservative proxy for Xet CDC."""
    chunks = []
    with open(path, 'rb') as f:
        while data := f.read(CHUNK_SIZE):
            chunks.append(hashlib.blake2b(data, digest_size=32).hexdigest())
    return chunks


def dir_chunks(d: str) -> dict:
    """Compute chunks for all Parquet files in a directory."""
    return {
        f: compute_chunks(os.path.join(d, f))
        for f in sorted(os.listdir(d)) if f.endswith('.parquet')
    }


def file_chunks(path: str) -> dict:
    """Single-file variant of dir_chunks."""
    return {os.path.basename(path): compute_chunks(path)}


def dir_delta(v1: dict, v2: dict) -> dict:
    """How many chunks in v2 aren't in v1? Returns reuse % and upload KB."""
    v1_all  = set(c for cs in v1.values() for c in cs)
    v2_all  = [c for cs in v2.values() for c in cs]
    new     = [c for c in v2_all if c not in v1_all]
    total   = len(v2_all)
    return {
        "total_chunks": total,
        "new_chunks":   len(new),
        "reuse_pct":    round(100 * (total - len(new)) / max(total, 1), 1),
        "upload_kb":    round(len(new) * CHUNK_SIZE / 1024, 0),
    }


def show(label: str, d: dict):
    bar = '█' * int(d['reuse_pct'] / 5) + '░' * (20 - int(d['reuse_pct'] / 5))
    print(f"  {label:48s} {bar} {d['reuse_pct']:5.1f}% reuse  ({d['upload_kb']:.0f} KB upload)")

## Dataset setup

Synthetic NLP dataset: 100,000 rows, 10 shards of 10,000 rows each. The append operation adds 1,000 new rows (1% growth) — a realistic incremental update.

In [3]:
N_TOTAL  = 100_000
N_SHARDS = 10
SHARD_SZ = N_TOTAL // N_SHARDS
N_NEW    = 1_000      # rows to append

rng = np.random.default_rng(42)

print("Generating base dataset...")
lengths      = rng.integers(30, 100, N_TOTAL)
all_word_ids = rng.integers(0, 8000, (N_TOTAL, 100))
texts = [
    f"doc_{i}: " + " ".join(f"w{all_word_ids[i, j]}" for j in range(int(lengths[i])))
    for i in range(N_TOTAL)
]
base_table = pa.table({
    "id":    pa.array(range(N_TOTAL),                        type=pa.int64()),
    "text":  pa.array(texts),
    "label": pa.array(rng.integers(0, 5, N_TOTAL).tolist(),  type=pa.int32()),
    "score": pa.array(rng.uniform(0, 1, N_TOTAL).tolist(),   type=pa.float32()),
})

print("Generating new rows to append...")
rng2         = np.random.default_rng(99)
new_lengths  = rng2.integers(30, 100, N_NEW)
new_word_ids = rng2.integers(0, 8000, (N_NEW, 100))
new_rows = pa.table({
    "id":    pa.array(range(N_TOTAL, N_TOTAL + N_NEW), type=pa.int64()),
    "text":  pa.array([
        f"doc_{N_TOTAL+i}: " + " ".join(f"w{new_word_ids[i,j]}" for j in range(int(new_lengths[i])))
        for i in range(N_NEW)
    ]),
    "label": pa.array(rng2.integers(0, 5, N_NEW).tolist(), type=pa.int32()),
    "score": pa.array(rng2.uniform(0, 1, N_NEW).tolist(),  type=pa.float32()),
})

print(f"Base:   {N_TOTAL:,} rows  {base_table.nbytes / 1e6:.1f} MB")
print(f"New:    {N_NEW:,} rows  {new_rows.nbytes / 1e6:.1f} MB")


def write_shards(table, directory, n_shards=N_SHARDS, compression='snappy', **kwargs):
    """Write table as N equal Parquet shards, mirroring HF Hub shard naming."""
    os.makedirs(directory, exist_ok=True)
    shard_sz = len(table) // n_shards
    for i in range(n_shards):
        start = i * shard_sz
        end   = start + shard_sz if i < n_shards - 1 else len(table)
        pq.write_table(
            table.slice(start, end - start),
            os.path.join(directory, f"train-{i:05d}-of-{n_shards:05d}.parquet"),
            compression=compression,
            **kwargs,
        )


# Verify pyarrow output is deterministic — required for chunk simulation to be valid
with tempfile.TemporaryDirectory() as tmp:
    shard = base_table.slice(0, SHARD_SZ)
    p1, p2 = os.path.join(tmp, "a.parquet"), os.path.join(tmp, "b.parquet")
    pq.write_table(shard, p1, compression='snappy')
    pq.write_table(shard, p2, compression='snappy')
    ok = open(p1, 'rb').read() == open(p2, 'rb').read()
    print(f"\n{'✓' if ok else '✗'} pyarrow output is {'deterministic' if ok else 'NOT deterministic'} — simulation is {'valid' if ok else 'unreliable'}")

Generating base dataset...


Generating new rows to append...
Base:   100,000 rows  40.8 MB
New:    1,000 rows  0.4 MB

✓ pyarrow output is deterministic — simulation is valid


---
## Approach 1: Reshard (current behavior)

Today, calling `push_to_hub` with a combined dataset rewrites everything from scratch. The library concatenates the old and new data, then shards it into N equal files. Even though 99% of the rows are unchanged, every shard gets different row ranges — so every chunk hash changes.

This is what a user gets today if they download their dataset, concatenate new rows, and call `push_to_hub` again.

In [4]:
combined = pa.concat_tables([base_table, new_rows])  # 101,000 rows

with tempfile.TemporaryDirectory() as tmp:
    d_v1     = os.path.join(tmp, "v1")
    d_reshard = os.path.join(tmp, "reshard")

    write_shards(base_table, d_v1)
    # Reshard: rewrite into 11 equal shards (rows redistribute across all files)
    write_shards(combined, d_reshard, n_shards=N_SHARDS + 1)

    v1_c      = dir_chunks(d_v1)
    reshard_c = dir_chunks(d_reshard)
    result_reshard = dir_delta(v1_c, reshard_c)

print(f"Append {N_NEW:,} rows to {N_TOTAL:,} row dataset — Approach 1: Reshard")
print()
show("Reshard (current push_to_hub behavior)", result_reshard)
print()
print("Every shard contains different row ranges → no chunk reuse.")
print(f"Upload: {result_reshard['upload_kb']:.0f} KB for {N_NEW} new rows + {N_TOTAL} unchanged rows.")

Append 1,000 rows to 100,000 row dataset — Approach 1: Reshard

  Reshard (current push_to_hub behavior)           ░░░░░░░░░░░░░░░░░░░░   0.0% reuse  (26752 KB upload)

Every shard contains different row ranges → no chunk reuse.
Upload: 26752 KB for 1000 new rows + 100000 unchanged rows.


---
## Approach 2: Add-shard (Wauplin)

Keep all 10 existing shards byte-for-byte. Write the new rows as an 11th shard. No existing shard is touched, so Xet sees zero new chunks for the first 10 files.

**Implementation in datasets**: on append, instead of calling `_push_parquet_shards_to_hub_single` for the full dataset, only write shard(s) for the new rows and use `CommitOperationAdd` for those files only — existing shards stay as-is in the repo.

**Tradeoff**: shards accumulate and become unequal over time. A compaction step would be needed periodically. Row count per shard file drifts — the 11th shard might have only 1,000 rows while others have 10,000.

In [5]:
with tempfile.TemporaryDirectory() as tmp:
    d_v1    = os.path.join(tmp, "v1")
    d_addshard = os.path.join(tmp, "addshard")

    write_shards(base_table, d_v1)

    # Add-shard: copy existing shards unchanged, write one new shard for new rows
    shutil.copytree(d_v1, d_addshard)
    pq.write_table(
        new_rows,
        os.path.join(d_addshard, f"train-{N_SHARDS:05d}-of-{N_SHARDS+1:05d}.parquet"),
        compression='snappy',
    )

    v1_c       = dir_chunks(d_v1)
    addshard_c = dir_chunks(d_addshard)
    result_addshard = dir_delta(v1_c, addshard_c)

print(f"Append {N_NEW:,} rows to {N_TOTAL:,} row dataset — Approach 2: Add-shard")
print()
show("Add-shard (10 shards untouched + 1 new)", result_addshard)
print()
print(f"Upload: {result_addshard['upload_kb']:.0f} KB — only the new shard. All existing data is free.")
print(f"Shard sizes: existing shards have {SHARD_SZ:,} rows each, new shard has {N_NEW:,} rows.")

Append 1,000 rows to 100,000 row dataset — Approach 2: Add-shard

  Add-shard (10 shards untouched + 1 new)          ███████████████████░  98.8% reuse  (320 KB upload)

Upload: 320 KB — only the new shard. All existing data is free.
Shard sizes: existing shards have 10,000 rows each, new shard has 1,000 rows.


---
## Approach 3: CDC-rewrite (lhoestq)

Rewrite the combined dataset as a single file using content-defined chunking for row group boundaries (`use_content_defined_chunking=True`, available in pyarrow ≥ 21).

**Why this helps**: Parquet writes row groups sequentially. When you append rows, the existing row groups at the start of the file produce the same bytes — their column chunk data is untouched. Only the new row groups and the updated file footer are genuinely new. Xet sees this as high chunk reuse.

**With CDC row groups** (pyarrow 21): row group boundaries are determined by content fingerprints, not by fixed row count thresholds. This makes the approach more robust to insertions *anywhere* in the dataset, not just appends.

**Without CDC** (pyarrow < 21): for pure append-at-end, the same prefix-preservation property holds with fixed row groups. The simulation below works either way.

**Implementation in datasets**: use `CommitOperationDelete` to remove the old shard files, then `CommitOperationAdd` to upload the new combined file. Hub sees one atomic commit.

In [6]:
cdc_kwargs = {"use_content_defined_chunking": True} if HAS_CDC else {}

if HAS_CDC:
    print(f"Using content_defined_chunking=True (pyarrow {pa.__version__})")
else:
    print(f"pyarrow {pa.__version__} — use_content_defined_chunking not available.")
    print("Writing with explicit row_group_size so existing row group bytes are preserved.\n")

# Row group size must match between v1 and v2 writes.
# With a consistent row_group_size, the first SHARD_SZ * N_SHARDS rows produce
# byte-identical column chunks in both files — pyarrow's deterministic output
# guarantees this. Only the final row group (new rows) + updated footer are new.
RG_SIZE = SHARD_SZ  # 10,000 rows per row group

with tempfile.TemporaryDirectory() as tmp:
    p_v1 = os.path.join(tmp, "v1.parquet")
    p_v2 = os.path.join(tmp, "v2.parquet")

    # v1: single file, 100K rows in 10 explicit row groups of 10K each
    pq.write_table(base_table, p_v1, compression='snappy',
                   row_group_size=RG_SIZE, **cdc_kwargs)

    # v2: combined dataset (100K + 1K) with the same row_group_size.
    # Row groups 0-9 produce identical bytes to v1; row group 10 (1K new rows)
    # and the updated footer are the only genuinely new bytes.
    pq.write_table(combined, p_v2, compression='snappy',
                   row_group_size=RG_SIZE, **cdc_kwargs)

    v1_c = file_chunks(p_v1)
    v2_c = file_chunks(p_v2)
    result_cdc = dir_delta(v1_c, v2_c)

    # Inspect where the byte streams diverge
    v1_list = list(v1_c.values())[0]
    v2_list = list(v2_c.values())[0]
    shared_prefix = sum(1 for a, b in zip(v1_list, v2_list) if a == b)

    v1_kb = os.path.getsize(p_v1) / 1024
    v2_kb = os.path.getsize(p_v2) / 1024

print(f"Append {N_NEW:,} rows to {N_TOTAL:,} row dataset — Approach 3: CDC-rewrite")
print()
show("CDC-rewrite (single file, prefix preserved)", result_cdc)
print()
print(f"File sizes:    v1={v1_kb:.0f} KB  v2={v2_kb:.0f} KB")
print(f"Chunk layout:  {len(v1_list)} chunks in v1, {len(v2_list)} in v2")
print(f"Shared prefix: first {shared_prefix} chunks byte-identical (existing row groups)")
print(f"Changed tail:  last {len(v2_list) - shared_prefix} chunks (new rows + footer)")

pyarrow 23.0.1 — use_content_defined_chunking not available.
Writing with explicit row_group_size so existing row group bytes are preserved.



Append 1,000 rows to 100,000 row dataset — Approach 3: CDC-rewrite

  CDC-rewrite (single file, prefix preserved)      ███████████████████░  98.8% reuse  (320 KB upload)

File sizes:    v1=26019 KB  v2=26286 KB
Chunk layout:  407 chunks in v1, 411 in v2
Shared prefix: first 406 chunks byte-identical (existing row groups)
Changed tail:  last 5 chunks (new rows + footer)


---
## Comparison

In [7]:
results = [
    ("Reshard (current behavior)",            result_reshard),
    ("Add-shard (Wauplin)",                   result_addshard),
    ("CDC-rewrite (lhoestq)",                 result_cdc),
]

print(f"{'Approach':<40}  {'Reuse':>8}  {'Upload':>10}  {'Notes'}")
print("-" * 85)
for label, r in results:
    note = ""
    if "Reshard" in label:   note = "all rows re-upload"
    if "Add-shard" in label: note = "uneven shards over time"
    if "CDC" in label:       note = "single file; pyarrow ≥ 21 for CDC"
    print(f"  {label:<40}  {r['reuse_pct']:>6.1f}%  {r['upload_kb']:>8.0f} KB  {note}")

print()
print(f"(Appending {N_NEW:,} rows to a {N_TOTAL:,}-row dataset.)")

if HAS_MPL:
    labels  = ["Reshard\n(current)", "Add-shard\n(Wauplin)", "CDC-rewrite\n(lhoestq)"]
    reuse   = [r['reuse_pct']  for _, r in results]
    colors  = ['#e74c3c', '#2ecc71', '#3498db']

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(labels, reuse, color=colors, alpha=0.85, width=0.5)
    ax.set_ylabel("Xet chunk reuse (%)")
    ax.set_ylim(0, 108)
    ax.set_title(f"push_to_hub(append=True): chunk reuse for +{N_NEW:,} rows")
    ax.axhline(100, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
    for bar, val in zip(bars, reuse):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f"{val:.0f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

Approach                                     Reuse      Upload  Notes
-------------------------------------------------------------------------------------
  Reshard (current behavior)                   0.0%     26752 KB  all rows re-upload
  Add-shard (Wauplin)                         98.8%       320 KB  uneven shards over time
  CDC-rewrite (lhoestq)                       98.8%       320 KB  single file; pyarrow ≥ 21 for CDC

(Appending 1,000 rows to a 100,000-row dataset.)


---
## Discussion

### Why Approach 1 is 0%

Resharding distributes rows differently across shards. If v1 has 10 shards of 10,000 rows and v2 has 11 shards of ~9,182 rows, every shard's byte content changes — the row ranges shift. Xet sees no familiar chunks.

### Why Approach 2 is ~99%

Existing shards are never written — they stay byte-identical in the repo. Xet doesn't touch them. Only the new shard (1,000 rows) uploads. Cost is proportional to new data only, not total dataset size.

**The catch**: after N appends, you have N+10 shards of unequal size. A compaction step (rewriting all shards to equal size) incurs a full upload at some point. The question is how often that's needed and who triggers it.

### Why Approach 3 is ~97–99%

Parquet files are written sequentially: [magic][row_group_0][row_group_1]…[row_group_N-1][footer]. When you append rows, the first N row groups are byte-identical to v1. Xet reuses all those chunks. Only the new row groups and the updated footer are novel bytes.

With CDC row groups (pyarrow ≥ 21): row group boundaries are content-defined, not fixed-size. This extends the benefit to insertions anywhere in the dataset, not just appends. Mid-dataset inserts shift only a bounded number of row groups.

**The catch**: the dataset is stored as a single file. Large files push on HF Hub file size limits and make partial reads less efficient. The library would need to manage the single-file ↔ multi-shard lifecycle.

### Tradeoffs

| | Add-shard | CDC-rewrite |
|--|-----------|-------------|
| **Xet reuse** | ~99% | ~97–99% |
| **Available today** | Yes | pyarrow ≥ 21 |
| **Shard count** | grows unbounded | stays at 1 |
| **Compaction needed** | yes, eventually | no |
| **Mid-dataset inserts** | full reshard | high reuse with CDC |
| **Hub file size limits** | distributed across shards | single large file |
| **Implementation complexity** | low | medium |

For a pure `append=True` use case (rows always added at the end), **Add-shard** is simpler to implement and available today. **CDC-rewrite** becomes more compelling when the operation is more general — arbitrary row insertions, not just end-appends — and once pyarrow 21 is widely available.

A hybrid: use Add-shard for `append=True` now, and offer `compact=True` to trigger a CDC-rewrite pass when the user is ready to pay the compaction cost.

---

*This prototype uses the same Xet simulation framework as the companion [parquet-xet-write-strategy](https://github.com/lex00/parquet-xet) notebook.*  
*See [datasets #6290](https://github.com/huggingface/datasets/issues/6290) for the full design discussion.*